In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
model=ChatOpenAI()

In [11]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    score: int

In [5]:
def create_outline(state: BlogState) -> BlogState:
    #fetch the topic from the state
    title=state['topic']
    #create a prompt for the model
    prompt=f"As an expert Content writer, Create a detailed outline for a blog post about - {title}."
    response = model.invoke(prompt).content
    #update the state with the outline
    state['outline'] = response

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:
    #fetch the outline from the state
    title=state['topic']
    outline=state['outline']
    #create a prompt for the model
    prompt=f"As an expert Content writer, Create a detailed blog post based on the topic:{title} and use the following outline -\n  {outline}."
    response = model.invoke(prompt).content
    #update the state with the content
    state['content'] = response

    return state

In [12]:
def evaluator (state: BlogState) -> BlogState:
    #fetch the content from the state
    content=state['content']
    outline=state['outline']
    #create a prompt for the model
    prompt=f"As an expert Content reviewer, Evaluate the following blog post and give it a score out of 10 based on its quality, clarity, and engagement. Provide only the score as an integer.\n\n{content}\n\nThe blog post should be evaluated based on the following outline:\n{outline}"
    response = model.invoke(prompt).content
    #update the state with the score
    state['score'] = int(response)

    return state

In [13]:
graph = StateGraph(BlogState)

#add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog )
graph.add_node('evaluator',evaluator )

#add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluator')
graph.add_edge('evaluator', END)

#compile the graph into a workflow
workflow = graph.compile()


In [14]:
initial_state ={'topic': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'topic': 'Rise of AI in India', 'outline': "I. Introduction \n    A. Definition of AI \n    B. Explanation of the rise of AI globally \n    C. Focus on the growth and adoption of AI technology in India \n\nII. Historical background of AI in India \n    A. Mention of early AI research and development in India \n    B. Examples of AI applications in Indian industries \n    C. Comparison of India's progress in AI with other countries \n\nIII. Current state of AI in India \n    A. Statistics on the growth of AI startups and investments in India \n    B. Government initiatives to promote AI research and development \n    C. Challenges faced by the AI industry in India \n\nIV. Impact of AI on Indian economy \n    A. Benefits of AI adoption in various industries \n    B. Job creation and skill development in the AI sector \n    C. Potential risks and concerns related to AI implementation \n\nV. Future prospects of AI in India \n    A. Predictions for the growth of AI market in India \n    B.

In [15]:
print(final_state['outline'])

I. Introduction 
    A. Definition of AI 
    B. Explanation of the rise of AI globally 
    C. Focus on the growth and adoption of AI technology in India 

II. Historical background of AI in India 
    A. Mention of early AI research and development in India 
    B. Examples of AI applications in Indian industries 
    C. Comparison of India's progress in AI with other countries 

III. Current state of AI in India 
    A. Statistics on the growth of AI startups and investments in India 
    B. Government initiatives to promote AI research and development 
    C. Challenges faced by the AI industry in India 

IV. Impact of AI on Indian economy 
    A. Benefits of AI adoption in various industries 
    B. Job creation and skill development in the AI sector 
    C. Potential risks and concerns related to AI implementation 

V. Future prospects of AI in India 
    A. Predictions for the growth of AI market in India 
    B. Opportunities for AI professionals and businesses 
    C. Recommen

In [16]:
print(final_state['content'])

The rise of Artificial Intelligence (AI) technology has been a game-changer in various industries around the world. From innovative business solutions to advancements in healthcare and education, AI has transformed the way we live and work. In recent years, India has also been witnessing a significant growth and adoption of AI technology, making it one of the key players in the global AI market.

I. Introduction

A. Definition of AI: AI refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, and self-correction.

B. Explanation of the rise of AI globally: The global AI market is expanding rapidly, with businesses investing in AI technology to gain a competitive edge and improve efficiency.

C. Focus on the growth and adoption of AI technology in India: India has seen a surge in the number of AI startups and investments in recent years, indicating a growing interest and demand for AI solutions in var

In [17]:
print(final_state['score'])

7
